In [1]:
import os
import pandas as pd
import numpy as np
import json

In [2]:
# extract important columns from judge json

In [3]:
def safe_json_loads(json_string):
    """
    Safely loads a JSON string, handling Markdown-style wrappers and errors.
    """
    if pd.isna(json_string) or json_string == '':
        return {}
    
    # 1. Strip the common Markdown wrapper for JSON code blocks
    # This specifically removes '```json' from the start and '```' from the end.
    cleaned_string = json_string.strip().replace('```json', '', 1).replace('```', '', 1)
    
    # 2. Strip any remaining leading/trailing whitespace
    cleaned_string = cleaned_string.strip()
    
    try:
        return json.loads(cleaned_string)
    except json.JSONDecodeError as e:
        # For debugging, you can print the error and the cleaned string
        # print(f"JSONDecodeError: {e} on string: '{cleaned_string}'")
        return {} # Return an empty dictionary

In [4]:
folder_paths = ["../output_data/advice_DD_gpt","../output_data/advice_reddit_gpt"]

In [5]:
# prepare DD annotations
DD_annotations = pd.read_csv("../output_data/daily_dilemmas_mfd_pro_anti_value.csv")

mapping = {'true ': True, 'false ': False}
DD_annotations['pro_value'] = DD_annotations['pro_value'].map(mapping)

DD_annotations["pro_value"].value_counts(dropna=False)

pro_value
True     83
False    56
NaN       8
Name: count, dtype: int64

In [6]:
for folder_path in folder_paths:
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_json(file_path, lines=True)
        
        df['GPT4o_eval_dict'] = df['GPT4o_eval'].apply(safe_json_loads)

        # Check for failed/empty rows
        failed_rows = df[df['GPT4o_eval_dict'].apply(len) == 0]

        # If there are failed rows, print the values from the 'GPT4o_eval' column
        if not failed_rows.empty:
            print(filename)
            print("\nValues in 'GPT4o_eval' for failed/empty rows:")
            print(failed_rows['GPT4o_eval'])
        
        df[['judge_explanation', 'judge_action_taken']] = df['GPT4o_eval_dict'].apply(pd.Series)

        # drop the intermediate column
        df = df.drop(columns=['GPT4o_eval_dict'])

        if folder_path == "../output_data/advice_DD_gpt":
            df = df.rename(columns={"mfd_value": "subscale"})
            df = df[["dilemma_idx", "item", "action", "subscale", "seed", "response", 'judge_explanation', 'judge_action_taken']]

            # change subscale of two dilemmas
            df.loc[df['dilemma_idx'].isin([4795,31247]), "subscale"] = "authority"

            # delete dilemmas
            idx_delete = [49737, 40653, 31029, 10634, 24902, 45075, 33509, 2600, 10328, 7631]
            df = df[~df['dilemma_idx'].isin(idx_delete)]

            # join with DD_annotations
            df = pd.merge(df, DD_annotations[["dilemma_idx", "action", "pro_value"]], on=["dilemma_idx", "action"], how='left')

        else:
            df = df.rename(columns={"dimension": "subscale", "pro-dimension": "pro_value"})
            # add unique identifier for each dilemma
            df["dilemma_idx"] = pd.factorize(df['item'])[0]
            df = df[["dilemma_idx", "item", "action", "subscale", "seed", "response", 'judge_explanation', 'judge_action_taken']]

        # save
        df.to_json(os.path.join("../output_data/advice", filename), orient="columns")